# Parte 2 – Modelo principal S1 e data augmentation (pessoa 2)

Nota: este notebook continua a partir do `Parte 1_testes.ipynb`. O código da pessoa 1 está copiado sem alterações em `parte2/parte1_base.py` (mesma preparação dos textos, mesmos pesos de classe, mesmas métricas e **as mesmas 5 divisões agrupadas**, seed 42). As funções novas estão em `parte2/s1.py`. Todos os resultados ficam na pasta `Parte2_resultados/` (um subdiretório por configuração, no mesmo formato da pasta `Test/` da pessoa 1).

Ordem: (1) ponto de partida, (2) classificador e C, (3) palavras vs fragmentos, (4) pesos das classes, (5) palavras-chave, (6) secções da nota, (7) pesos por campo, (8) augmentation, (9) configuração final, treino com todo o treino e modelo guardado.
Em cada passo a melhor configuração passa para o passo seguinte, escolhida com o critério `priority()` da pessoa 1 (cumprir accuracy > 0.49 e F1 mínimo ≥ 0.25 → maior macro-F1 → maior accuracy). O teste nunca é usado.

In [ ]:
import sys, json
sys.path.insert(0, 'parte2')
import pandas as pd, numpy as np
from threadpoolctl import threadpool_limits
from parte1_base import data, splits, specialties
from s1 import *

pd.set_option('display.width', 200)
print(len(data), 'cases,', data['group_id'].nunique(), 'groups,', [len(s['evaluation']) for s in splits], 'validation cases per fold')

## 1. Ponto de partida
Melhor modelo da pessoa 1 (`all_fields_balanced_class_weights`: SVM, C=0.5, palavras em todos os campos + fragmentos 3–5 nos campos curtos com peso 0.5, pesos de classe q=1). A pessoa 1 obteve accuracy 0.591 / macro-F1 0.623 / F1 mín. 0.447; aqui é recalculado com o código do S1 para confirmar que os números coincidem.

In [ ]:
best = {'classifier': 'svm', 'c': 0.5, 'q': 1, 'keywords': 'strip', 'sections': False, 'weights': default_weights('both', 0.5)}
with threadpool_limits(limits=4):
    t1 = run({'start_person1_best': best}, 'step1_start.csv')
t1

## 2. Classificador e parâmetro C
C é o inverso da força de regularização (C pequeno = modelo mais simples). Comparamos SVM linear e regressão logística (LR) com vários C.

In [ ]:
configs = {f'svm_C{c}': {**best, 'classifier': 'svm', 'c': c} for c in [0.1, 0.5, 1]}
configs.update({f'lr_C{c}': {**best, 'classifier': 'lr', 'c': c} for c in [3, 10, 30]})
with threadpool_limits(limits=4):
    t2 = run(configs, 'step2_classifier.csv')
best = configs[best_of(t2)]
print('chosen:', best_of(t2)); t2

## 3. Palavras vs fragmentos de palavras
`word`: só palavras (uni+bigramas); `char`: só fragmentos 3–5 caracteres; `both`: palavras + fragmentos nos campos curtos (como a pessoa 1); `both+note_char`: também fragmentos na nota clínica.

In [ ]:
configs = {
    'repr_word': {**best, 'weights': default_weights('word')},
    'repr_char': {**best, 'weights': default_weights('char')},
    'repr_both': {**best, 'weights': default_weights('both', 0.5)},
    'repr_both_note_char': {**best, 'weights': default_weights('both', 0.5, transcription_char=True)},
}
with threadpool_limits(limits=4):
    t3 = run(configs, 'step3_representation.csv')
best = configs[best_of(t3)]
print('chosen:', best_of(t3)); t3

## 4. Pesos das classes
Peso de cada caso = (N / (12 · n_classe))^q (função `class_weights` da pessoa 1). q=0: sem pesos; q=1: balanceado; q=0.5: intermédio.

In [ ]:
configs = {f'class_weights_q{q}': {**best, 'q': q} for q in [0, 0.5, 1]}
with threadpool_limits(limits=4):
    t4 = run(configs, 'step4_class_weights.csv')
best = configs[best_of(t4)]
print('chosen:', best_of(t4)); t4

## 5. Tratamento das palavras-chave
`strip`: retira a 1.ª keyword quando é o nome da especialidade (fuga da label); `none`: sem keywords; `mask`: substitui nomes de especialidades por "specialty" em todos os campos; `raw`: keywords originais, **com a fuga** — mostrado só para medir o efeito da fuga, não entra na escolha.

In [ ]:
configs = {f'keywords_{k}': {**best, 'keywords': k} for k in ['strip', 'none', 'mask', 'raw']}
with threadpool_limits(limits=4):
    t5 = run(configs, 'step5_keywords.csv')
best = configs[best_of(t5.drop(index='keywords_raw'))]
print('chosen:', best_of(t5.drop(index='keywords_raw'))); t5

## 6. Seleção de secções da nota
`select_sections` (pessoa 1) mantém só as secções DIAGNOSIS, IMPRESSION, ASSESSMENT, PROCEDURE, FINDINGS e PLAN da nota clínica.

In [ ]:
configs = {'sections_full_note': {**best, 'sections': False}, 'sections_selected': {**best, 'sections': True}}
with threadpool_limits(limits=4):
    t6 = run(configs, 'step6_sections.csv')
best = configs[best_of(t6)]
print('chosen:', best_of(t6)); t6

## 7. Pesos por campo
Cada bloco (campo × palavras/fragmentos) é multiplicado por um peso; procura por subida coordenada em {0, 0.5, 1, 2}, avaliada nas 5 divisões com `priority()`. Peso 0 retira o bloco.

In [ ]:
with threadpool_limits(limits=4):
    fw, fw_log = field_weight_search(best)
configs = {'field_weights_before': best, 'field_weights_after': {**best, 'weights': fw}}
with threadpool_limits(limits=4):
    t7 = run(configs, 'step7_field_weights.csv')
best = configs[best_of(t7)]
print('chosen:', best_of(t7)); display(fw_log); t7

## 8. Data augmentation
Aplicada **só ao treino de cada divisão** (as variantes herdam o grupo do caso original, por isso nunca passam para a validação).
- `none`: treino original;
- `target` (classes pequenas, <100 casos no treino da divisão: Neurosurgery, Ophthalmology, Psychiatry-Psychology, Dermatology): 1 variante por *back-translation* EN→DE→EN (OPUS-MT) + 1 variante com 60% das secções da nota;
- `all` (várias classes): 1 variante com 60% das secções para cada caso de todas as classes + 30% dos casos com keywords copiados sem keywords;
- `copy` (controlo): cópias exatas de casos das classes pequenas, o mesmo número que `target` acrescenta.

In [ ]:
configs = {f'augmentation_{a}': {**best, 'augmentation': a} for a in ['none', 'target', 'all', 'copy']}
with threadpool_limits(limits=4):
    t8 = run(configs, 'step8_augmentation.csv')
best = configs[best_of(t8)]
print('chosen:', best_of(t8)); t8

### 8.1 Contagem e revisão das variantes
Para a divisão 0: número de variantes por tipo e por especialidade, semelhança com o texto original (Jaccard de palavras) e exemplos. Os registos completos ficam em `Parte2_resultados/augmentation_log_<modo>.csv`.

In [ ]:
def jaccard(a, b):
    a, b = set(str(a).lower().split()), set(str(b).lower().split())
    return len(a & b) / max(1, len(a | b))

train0 = data[data['row_id'].isin(splits[0]['train'])]
for mode in ['target', 'all', 'copy']:
    _, log = augment(train0, mode, seed=0)
    original = data.set_index('row_id').loc[log['source_row_id']]
    log = log.assign(original_description=original['description'].values,
                     original_transcription=original['transcription'].values)
    log['similarity_transcription'] = [jaccard(a, b) for a, b in zip(log['transcription'], log['original_transcription'])]
    log['similarity_description'] = [jaccard(a, b) for a, b in zip(log['description'], log['original_description'])]
    log[['source_row_id', 'medical_specialty', 'augmentation', 'similarity_description', 'similarity_transcription',
         'original_description', 'description', 'keywords', 'original_transcription', 'transcription']].to_csv(OUT / f'augmentation_log_{mode}.csv', index=False)
    print(f'\n=== {mode}: {len(log)} variants (fold 0) ===')
    print(pd.crosstab(log['medical_specialty'], log['augmentation']))
    print(log.groupby('augmentation')[['similarity_description', 'similarity_transcription']].mean().round(3))

In [ ]:
bt = pd.read_csv(OUT / 'augmentation_log_target.csv').query("augmentation == 'backtranslation'")
for _, r in bt.sort_values('similarity_description').head(4).iterrows():
    print(r['medical_specialty'], '| similarity', round(r['similarity_description'], 2))
    print('  original :', r['original_description'][:200])
    print('  variant  :', r['description'][:200], '\n')

## 9. Configuração final, treino com todo o treino e modelo guardado

In [ ]:
import joblib
summary = pd.concat([t1, t2, t3, t4, t5, t6, t7, t8]).drop_duplicates()
summary.to_csv(OUT / 'all_results.csv')
final = dict(best)
print('Final S1 configuration:'); print(json.dumps({k: v for k, v in final.items()}, indent=1))

q = final.pop('q'); aug = final.pop('augmentation', 'none')
train_all, log_all = augment(data, aug, seed=42)
model = create_s1_model(**final)
model.fit(train_all, train_all['medical_specialty'], classifier__sample_weight=class_weights(train_all['medical_specialty'].tolist(), q))
joblib.dump(model, OUT / 's1_final_model.joblib')
(OUT / 's1_final_config.json').write_text(json.dumps({**best, 'trained_on_cases': len(data), 'augmented_cases': len(log_all)}, indent=1))
print('saved', OUT / 's1_final_model.joblib', '| trained on', len(train_all), 'rows')

In [ ]:
# Sanity check for person 3: load the saved model and predict the test set (results are NOT evaluated here)
from parte1_base import load_test
from pathlib import Path
test = load_test(Path('StudentsPack') / 'test_no_labels.csv')
loaded = joblib.load(OUT / 's1_final_model.joblib')
pred = loaded.predict(test)
print(len(test), 'test cases,', len(pred), 'predictions'); print(pd.Series(pred).value_counts())